# Dark Manifold V32: Environment-Aware Steady-State Simulation

**Combines the best of V19 + V31:**

| Feature | V31 | V32 |
|---------|-----|-----|
| Open system (boundary fluxes) | ✅ | ✅ |
| Dynamic perturbation vectors | ❌ | ✅ |
| Environment-responsive kinetics | ❌ | ✅ |
| Stressor types | None | Heat, Cold, Hypoxia, Oxidative, Starvation |

## Perturbation Vector (6 dimensions)
```
[temperature, oxygen, oxidative_stress, carbon_availability, pH, time]
```

## Stressor Effects on Metabolism
| Stressor | Primary Effect | Metabolic Response |
|----------|---------------|-------------------|
| Heat shock | Protein denaturation | ↑ ATP demand, ↓ enzyme activity |
| Cold shock | Membrane rigidity | ↓ Transport rates |
| Hypoxia | O₂ limitation | ↓ ETC, ↑ Glycolysis (Warburg) |
| Oxidative | ROS damage | ↑ NADPH demand, ↓ NAD+ |
| Starvation | Glucose depletion | ↓ Glycolysis, ↑ Gluconeogenesis |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import time
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Environment-Aware Metabolic Network

In [ ]:
# Perturbation vector definition
PERTURBATION_DIMS = [
    'temperature',      # -1 (cold) to +1 (heat), 0 = normal (37°C)
    'oxygen',           # 0 (anoxia) to 1 (normoxia)
    'oxidative_stress', # 0 (none) to 1 (severe ROS)
    'carbon',           # 0 (starvation) to 1 (glucose-rich)
    'ph',               # -1 (acidic) to +1 (basic), 0 = 7.0
    'time_normalized',  # 0 to 1 (for temporal context)
]
N_PERTURBATION = len(PERTURBATION_DIMS)

print(f"Perturbation dimensions: {N_PERTURBATION}")
for i, dim in enumerate(PERTURBATION_DIMS):
    print(f"  [{i}] {dim}")

In [ ]:
class EnvironmentAwareNetwork:
    """
    Open metabolic network with environment-dependent kinetics.
    
    Enzyme rates are modulated by perturbation vector:
    - Temperature affects all enzyme kcat (Q10 effect)
    - Oxygen affects ETC and oxidative phosphorylation
    - Oxidative stress damages NAD+ pool
    - Carbon availability affects glucose uptake
    """
    
    def __init__(self):
        self.metabolites = [
            'glc', 'g6p', 'f6p', 'fbp', 'gap', 'bpg', '3pg', '2pg', 'pep', 'pyr',
            'lac', 'acoa',
            'atp', 'adp', 'amp',
            'nad', 'nadh',
            'pi',
            'ros',  # Reactive oxygen species (stress marker)
        ]
        self.n_met = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Reaction definitions with environment sensitivity
        # Format: (name, subs, prods, kcat_base, Km, env_sensitivity)
        # env_sensitivity: dict of {perturbation_idx: effect_coefficient}
        self.reactions = [
            # === BOUNDARY FLUXES ===
            # Glucose uptake: depends on external carbon
            ('GLUT', [], [(0, 1)], 2.5, 1.0, {3: 1.5}),  # carbon ↑ → uptake ↑
            # Lactate export
            ('MCT', [(10, 1)], [], 5.0, 2.0, {}),
            # ETC (NAD+ regeneration): depends on oxygen
            ('ETC', [(16, 1)], [(15, 1)], 15.0, 0.1, {1: 2.0}),  # O2 ↑ → ETC ↑
            # ROS production: increases with oxidative stress
            ('ROS_gen', [], [(18, 1)], 0.1, 1.0, {2: 5.0}),  # oxidative ↑ → ROS ↑
            # ROS clearance (antioxidants)
            ('ROS_clear', [(18, 1)], [], 2.0, 0.5, {}),
            
            # === GLYCOLYSIS (temperature-sensitive) ===
            ('HK', [(0, 1), (12, 1)], [(1, 1), (13, 1)], 100.0, 0.1, {0: 0.3}),
            ('PGI', [(1, 1)], [(2, 1)], 600.0, 0.3, {0: 0.2}),
            ('PFK', [(2, 1), (12, 1)], [(3, 1), (13, 1)], 100.0, 0.05, {0: 0.3, 4: -0.2}),  # pH sensitive
            ('ALDO', [(3, 1)], [(4, 2)], 40.0, 0.05, {0: 0.2}),
            ('GAPDH', [(4, 1), (15, 1), (17, 1)], [(5, 1), (16, 1)], 200.0, 0.05, {0: 0.3, 2: -0.3}),  # ROS-sensitive
            ('PGK', [(5, 1), (13, 1)], [(6, 1), (12, 1)], 400.0, 0.3, {0: 0.2}),
            ('PGM', [(6, 1)], [(7, 1)], 500.0, 0.5, {0: 0.2}),
            ('ENO', [(7, 1)], [(8, 1)], 300.0, 0.3, {0: 0.2}),
            ('PK', [(8, 1), (13, 1)], [(9, 1), (12, 1)], 200.0, 0.2, {0: 0.3}),
            
            # === PYRUVATE FATE ===
            ('LDH', [(9, 1), (16, 1)], [(10, 1), (15, 1)], 300.0, 0.2, {1: -0.5}),  # Hypoxia ↑ → LDH ↑
            ('PDH', [(9, 1), (15, 1)], [(11, 1), (16, 1)], 20.0, 0.1, {1: 0.5}),   # O2 ↑ → PDH ↑
            
            # === ATP HOMEOSTASIS ===
            ('ATPase', [(12, 1)], [(13, 1), (17, 1)], 60.0, 1.0, {0: 0.5}),  # Heat ↑ → ATP demand ↑
            ('ADK', [(12, 1), (14, 1)], [(13, 2)], 500.0, 0.5, {}),
        ]
        self.n_rxn = len(self.reactions)
        
        # Build matrices
        self.S = np.zeros((self.n_met, self.n_rxn))
        self.kcat_base = np.zeros(self.n_rxn)
        self.Km = np.zeros(self.n_rxn)
        self.env_sensitivity = []  # List of dicts
        
        for j, rxn in enumerate(self.reactions):
            name, subs, prods, kcat, km, env_sens = rxn
            self.kcat_base[j] = kcat
            self.Km[j] = km
            self.env_sensitivity.append(env_sens)
            for idx, coef in subs:
                self.S[idx, j] -= coef
            for idx, coef in prods:
                self.S[idx, j] += coef
        
        # Initial concentrations
        self.M0 = np.array([
            2.0,   # glc
            0.08,  # g6p
            0.03,  # f6p  
            0.02,  # fbp
            0.04,  # gap
            0.002, # bpg
            0.06,  # 3pg
            0.01,  # 2pg
            0.02,  # pep
            0.08,  # pyr
            1.0,   # lac
            0.02,  # acoa
            3.0,   # atp
            0.5,   # adp
            0.1,   # amp
            0.8,   # nad
            0.3,   # nadh
            5.0,   # pi
            0.01,  # ros (low baseline)
        ])

network = EnvironmentAwareNetwork()
print(f"Network: {network.n_met} metabolites, {network.n_rxn} reactions")

## 2. Stressor Profiles

In [ ]:
def create_stressor_profile(stressor_type, times, intensity=1.0):
    """
    Create time-varying perturbation vector for a given stressor.
    
    Returns: (n_timepoints, 6) array
    [temperature, oxygen, oxidative_stress, carbon, ph, time_normalized]
    """
    n_t = len(times)
    t_max = times[-1]
    P = np.zeros((n_t, N_PERTURBATION))
    
    # Time normalization
    P[:, 5] = times / t_max
    
    # Baseline (normal conditions)
    P[:, 0] = 0.0   # Normal temperature
    P[:, 1] = 1.0   # Normal oxygen
    P[:, 2] = 0.0   # No oxidative stress
    P[:, 3] = 1.0   # Normal carbon
    P[:, 4] = 0.0   # Normal pH
    
    if stressor_type == 'normal':
        pass  # Keep baseline
        
    elif stressor_type == 'heat_shock':
        # Sudden heat shock that decays
        tau = t_max / 4
        P[:, 0] = intensity * np.exp(-times / tau)
        P[:, 2] = 0.3 * intensity * np.exp(-times / tau)  # Heat causes some ROS
        
    elif stressor_type == 'heat_sustained':
        # Sustained elevated temperature
        P[:, 0] = intensity * 0.7  # Constant heat
        P[:, 2] = 0.2 * intensity  # Chronic mild oxidative stress
        
    elif stressor_type == 'cold_shock':
        # Cold shock
        tau = t_max / 3
        P[:, 0] = -intensity * np.exp(-times / tau)  # Negative = cold
        
    elif stressor_type == 'hypoxia':
        # Gradual oxygen depletion
        P[:, 1] = 1.0 - intensity * (1 - np.exp(-times / (t_max/2)))
        P[:, 1] = np.clip(P[:, 1], 0.05, 1.0)  # Min 5% O2
        
    elif stressor_type == 'hypoxia_reoxygenation':
        # Hypoxia then sudden reoxygenation (causes ROS burst)
        mid = n_t // 2
        P[:mid, 1] = 0.1 * intensity  # Low O2
        P[mid:, 1] = 1.0  # Sudden reoxygenation
        # ROS burst on reoxygenation
        reox_times = times[mid:] - times[mid]
        P[mid:, 2] = intensity * np.exp(-reox_times / (t_max/10))
        
    elif stressor_type == 'oxidative_stress':
        # Direct oxidative stress (e.g., H2O2 treatment)
        tau = t_max / 3
        P[:, 2] = intensity * np.exp(-times / tau)
        
    elif stressor_type == 'starvation':
        # Glucose depletion
        P[:, 3] = 1.0 - intensity * (1 - np.exp(-times / (t_max/3)))
        P[:, 3] = np.clip(P[:, 3], 0.1, 1.0)
        
    elif stressor_type == 'starvation_refeed':
        # Starvation then refeeding
        mid = n_t // 2
        P[:mid, 3] = 0.1  # Starved
        P[mid:, 3] = 1.0  # Refed
        
    elif stressor_type == 'acidosis':
        # Acidic pH stress
        P[:, 4] = -intensity * 0.5  # Acidic
        
    elif stressor_type == 'multi_stress':
        # Combined stress: heat + hypoxia + oxidative
        tau = t_max / 4
        P[:, 0] = 0.5 * intensity * np.exp(-times / tau)  # Mild heat
        P[:, 1] = 1.0 - 0.5 * intensity * (1 - np.exp(-times / (t_max/2)))  # Mild hypoxia
        P[:, 2] = 0.5 * intensity * np.exp(-times / tau)  # Mild oxidative
    
    return P

# List available stressors
STRESSOR_TYPES = [
    'normal', 'heat_shock', 'heat_sustained', 'cold_shock',
    'hypoxia', 'hypoxia_reoxygenation', 'oxidative_stress',
    'starvation', 'starvation_refeed', 'acidosis', 'multi_stress'
]
print(f"Available stressors: {len(STRESSOR_TYPES)}")
for s in STRESSOR_TYPES:
    print(f"  • {s}")

In [ ]:
# Visualize stressor profiles
times = np.linspace(0, 20, 100)  # 20 min simulation

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for ax, stressor in zip(axes.flat, STRESSOR_TYPES + ['']):
    if stressor == '':
        ax.axis('off')
        continue
    
    P = create_stressor_profile(stressor, times)
    
    ax.plot(times, P[:, 0], 'r-', label='Temp', lw=2)
    ax.plot(times, P[:, 1], 'c-', label='O₂', lw=2)
    ax.plot(times, P[:, 2], 'm-', label='ROS', lw=2)
    ax.plot(times, P[:, 3], 'g-', label='Carbon', lw=2)
    ax.plot(times, P[:, 4], 'orange', label='pH', lw=2)
    
    ax.set_title(stressor.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('Time (min)')
    ax.set_ylim(-1.2, 1.5)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)

plt.suptitle('V32 Stressor Profiles — Dynamic Perturbation Vectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v32_stressor_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Environment-Modulated Model

In [ ]:
class EnvironmentAwareModel(nn.Module):
    """
    Model where enzyme kinetics are modulated by perturbation vector.
    
    kcat_effective = kcat_base * exp(sum(sensitivity * perturbation))
    """
    
    def __init__(self, network):
        super().__init__()
        
        self.n_met = network.n_met
        self.n_rxn = network.n_rxn
        self.n_pert = N_PERTURBATION
        
        self.register_buffer('S', torch.tensor(network.S, dtype=torch.float32))
        self.register_buffer('kcat_base', torch.tensor(network.kcat_base, dtype=torch.float32))
        self.register_buffer('Km', torch.tensor(network.Km, dtype=torch.float32))
        self.register_buffer('M0', torch.tensor(network.M0, dtype=torch.float32))
        
        # Build environment sensitivity matrix
        # Shape: (n_rxn, n_perturbation)
        env_sens = torch.zeros(self.n_rxn, self.n_pert)
        for j, sens_dict in enumerate(network.env_sensitivity):
            for pert_idx, coef in sens_dict.items():
                env_sens[j, pert_idx] = coef
        self.register_buffer('env_sensitivity', env_sens)
        
        # Build substrate mask
        substrate_mask = torch.zeros(self.n_rxn, self.n_met)
        for j, rxn in enumerate(network.reactions):
            _, subs, _, _, _, _ = rxn
            for idx, _ in subs:
                substrate_mask[j, idx] = 1.0
        self.register_buffer('substrate_mask', substrate_mask)
        
        self.min_conc = 1e-9
        self.max_conc = 50.0
    
    def compute_effective_kcat(self, P):
        """
        Compute environment-modulated kcat.
        
        P: (batch, n_perturbation) or (n_perturbation,)
        Returns: (batch, n_rxn) or (n_rxn,)
        """
        if P.dim() == 1:
            P = P.unsqueeze(0)
        
        # Modulation: exp(sensitivity @ perturbation)
        # env_sensitivity: (n_rxn, n_pert), P: (batch, n_pert)
        modulation = torch.exp(torch.matmul(P, self.env_sensitivity.T))  # (batch, n_rxn)
        modulation = torch.clamp(modulation, 0.1, 10.0)  # Limit extreme modulation
        
        return self.kcat_base * modulation
    
    def compute_flux(self, M, P):
        """Compute fluxes with environment modulation."""
        batch_size = M.shape[0]
        kcat_eff = self.compute_effective_kcat(P)  # (batch, n_rxn)
        
        v = torch.zeros(batch_size, self.n_rxn, device=M.device)
        
        for j in range(self.n_rxn):
            sub_indices = (self.substrate_mask[j] > 0).nonzero(as_tuple=True)[0]
            
            if len(sub_indices) == 0:
                # Pure source reaction
                v[:, j] = kcat_eff[:, j]
            else:
                # Michaelis-Menten with product of saturations
                sat = torch.ones(batch_size, device=M.device)
                for idx in sub_indices:
                    sat = sat * M[:, idx] / (self.Km[j] + M[:, idx] + 1e-10)
                v[:, j] = kcat_eff[:, j] * sat
        
        return v
    
    def forward(self, M, P, dt):
        """Single step with environment."""
        M = torch.clamp(M, min=self.min_conc)
        v = self.compute_flux(M, P)
        dM_dt = torch.matmul(v, self.S.T)
        M_next = M + dM_dt * dt
        return torch.clamp(M_next, min=self.min_conc, max=self.max_conc), v
    
    @torch.no_grad()
    def simulate_with_perturbation(self, P_trajectory, n_steps_per_point, dt, save_every=100):
        """
        Simulate with time-varying perturbation.
        
        P_trajectory: (n_timepoints, n_perturbation) - perturbation at each saved point
        n_steps_per_point: integration steps between saved points
        """
        n_points = len(P_trajectory)
        M = self.M0.unsqueeze(0).to(self.S.device)
        P_traj = torch.tensor(P_trajectory, dtype=torch.float32, device=self.S.device)
        
        M_hist = torch.zeros(n_points, self.n_met, device=M.device)
        v_hist = torch.zeros(n_points, self.n_rxn, device=M.device)
        M_hist[0] = M[0]
        
        for t_idx in range(1, n_points):
            P_current = P_traj[t_idx].unsqueeze(0)
            
            # Integrate from t_idx-1 to t_idx
            for _ in range(n_steps_per_point):
                M, v = self.forward(M, P_current, dt)
            
            M_hist[t_idx] = M[0]
            v_hist[t_idx] = v[0]
        
        return M_hist, v_hist

model = EnvironmentAwareModel(network).to(device)
print(f"Model ready on {device}")

## 4. Run Simulation Under Different Stressors

In [ ]:
# Configuration
VIRTUAL_DURATION_MIN = 20
N_SAVED_POINTS = 500
STEPS_PER_POINT = 1000
dt = VIRTUAL_DURATION_MIN / (N_SAVED_POINTS * STEPS_PER_POINT)

times = np.linspace(0, VIRTUAL_DURATION_MIN, N_SAVED_POINTS)

print("="*70)
print("DARK MANIFOLD V32 — ENVIRONMENT-AWARE SIMULATION")
print("="*70)
print(f"Duration: {VIRTUAL_DURATION_MIN} min | Points: {N_SAVED_POINTS}")
print(f"Steps per point: {STEPS_PER_POINT} | dt: {dt*60*1e6:.1f} µs")
print("="*70)

In [ ]:
# Run simulations under different stressors
STRESSORS_TO_TEST = ['normal', 'heat_shock', 'hypoxia', 'oxidative_stress', 'starvation', 'multi_stress']

results = {}

for stressor in STRESSORS_TO_TEST:
    print(f"\n▶ Simulating: {stressor}")
    
    # Create perturbation profile
    P = create_stressor_profile(stressor, times)
    
    # Run simulation
    start = time.time()
    M_hist, v_hist = model.simulate_with_perturbation(
        P, 
        n_steps_per_point=STEPS_PER_POINT,
        dt=dt
    )
    elapsed = time.time() - start
    
    # Store results
    M_np = M_hist.cpu().numpy()
    v_np = v_hist.cpu().numpy()
    
    results[stressor] = {
        'M': M_np,
        'v': v_np,
        'P': P,
        'time': elapsed,
    }
    
    # Quick stats
    atp_final = M_np[-1, network.met_idx['atp']]
    lac_final = M_np[-1, network.met_idx['lac']]
    ros_final = M_np[-1, network.met_idx['ros']]
    print(f"   Time: {elapsed:.1f}s | ATP: {atp_final:.2f} | Lac: {lac_final:.2f} | ROS: {ros_final:.3f}")

print("\n✅ All simulations complete!")

## 5. Compare Metabolic Responses to Stressors

In [ ]:
# Compare key metabolites across stressors
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metabolites_to_plot = [
    ('atp', 'ATP'),
    ('lac', 'Lactate'),
    ('ros', 'ROS'),
    ('nadh', 'NADH'),
    ('glc', 'Glucose'),
    ('pyr', 'Pyruvate'),
]

colors = plt.cm.tab10(np.linspace(0, 1, len(STRESSORS_TO_TEST)))

for ax, (met_key, met_name) in zip(axes.flat, metabolites_to_plot):
    met_idx = network.met_idx[met_key]
    
    for i, stressor in enumerate(STRESSORS_TO_TEST):
        M = results[stressor]['M']
        ax.plot(times, M[:, met_idx], color=colors[i], label=stressor, lw=1.5)
    
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Concentration (mM)')
    ax.set_title(met_name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('V32: Metabolic Response to Different Stressors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v32_stressor_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Energy charge comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy charge over time
ax = axes[0]
for i, stressor in enumerate(STRESSORS_TO_TEST):
    M = results[stressor]['M']
    atp = M[:, network.met_idx['atp']]
    adp = M[:, network.met_idx['adp']]
    amp = M[:, network.met_idx['amp']]
    ec = (atp + 0.5 * adp) / (atp + adp + amp + 0.01)
    ax.plot(times, ec, color=colors[i], label=stressor, lw=1.5)

ax.axhline(0.85, color='g', ls='--', alpha=0.7, label='Optimal')
ax.axhline(0.7, color='orange', ls='--', alpha=0.7, label='Stress threshold')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Charge')
ax.set_title('Energy Charge Under Stress')
ax.set_ylim([0.3, 1.0])
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Final state bar chart
ax = axes[1]
final_atp = [results[s]['M'][-1, network.met_idx['atp']] for s in STRESSORS_TO_TEST]
final_lac = [results[s]['M'][-1, network.met_idx['lac']] for s in STRESSORS_TO_TEST]
final_ros = [results[s]['M'][-1, network.met_idx['ros']] * 10 for s in STRESSORS_TO_TEST]  # Scale ROS

x = np.arange(len(STRESSORS_TO_TEST))
width = 0.25
ax.bar(x - width, final_atp, width, label='ATP', color='blue', alpha=0.7)
ax.bar(x, final_lac, width, label='Lactate', color='orange', alpha=0.7)
ax.bar(x + width, final_ros, width, label='ROS (×10)', color='red', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in STRESSORS_TO_TEST], fontsize=8)
ax.set_ylabel('Concentration (mM)')
ax.set_title('Final State Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('v32_energy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Table

In [ ]:
print("\n" + "="*80)
print("V32 SIMULATION RESULTS SUMMARY")
print("="*80)
print(f"{'Stressor':<20} {'ATP':>8} {'Lactate':>10} {'ROS':>8} {'EC':>8} {'Status':<15}")
print("-"*80)

for stressor in STRESSORS_TO_TEST:
    M = results[stressor]['M']
    atp = M[-1, network.met_idx['atp']]
    adp = M[-1, network.met_idx['adp']]
    amp = M[-1, network.met_idx['amp']]
    lac = M[-1, network.met_idx['lac']]
    ros = M[-1, network.met_idx['ros']]
    ec = (atp + 0.5 * adp) / (atp + adp + amp + 0.01)
    
    if ec > 0.8:
        status = "✅ Healthy"
    elif ec > 0.6:
        status = "⚠️ Stressed"
    else:
        status = "❌ Critical"
    
    print(f"{stressor:<20} {atp:>8.3f} {lac:>10.3f} {ros:>8.4f} {ec:>8.3f} {status:<15}")

print("="*80)

## 7. Save Results

In [ ]:
# Save comprehensive results
output = {
    'metadata': {
        'model': 'Dark Manifold V32',
        'description': 'Environment-aware open system with dynamic stressors',
        'virtual_duration_min': VIRTUAL_DURATION_MIN,
        'n_metabolites': network.n_met,
        'n_reactions': network.n_rxn,
        'perturbation_dims': PERTURBATION_DIMS,
        'stressors_tested': STRESSORS_TO_TEST,
    },
    'metabolite_names': network.metabolites,
    'time_min': times.tolist(),
    'results': {}
}

for stressor in STRESSORS_TO_TEST:
    M = results[stressor]['M']
    output['results'][stressor] = {
        'final_state': {name: float(M[-1, i]) for name, i in network.met_idx.items()},
        'compute_time': results[stressor]['time'],
    }

with open('dark_manifold_v32_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("✅ Saved to dark_manifold_v32_results.json")